# 2026 项目 · B · 吉伦特省的住房市场（DVF 2024）

小组：`NN` · 成员：... · 主问题：...

请先阅读 `PROJECT_2026_instructions.ipynb`。这个 notebook 为你提供数据和结构；请替换每一处 *TODO*。

## 数据

- `dvf_gironde_2024.csv.gz` — DGFiP，*Demandes de valeurs foncières géolocalisées*（带地理坐标的房地产交易申报，Licence Ouverte 开放许可）：2024 年
  在吉伦特省（33）登记备案的每一笔房产交易。一行 = 一笔交易的**一部分**（一个分摊单元、一块地、一栋建筑）。
  主要列：`id_mutation`（交易 id）、`date_mutation`、`nature_mutation`（出售、互换……）、`valeur_fonciere`
  （价格，单位 €，**在同一笔交易的每一行上重复出现**）、`type_local`（独栋住宅、公寓、附属建筑、商业用房）、
  `surface_reelle_bati`（m²）、`nombre_pieces_principales`、`surface_terrain`、`nom_commune`、`longitude`、`latitude`。
- 文档：https://www.data.gouv.fr/fr/datasets/demandes-de-valeurs-foncieres-geolocalisees/

In [ ]:
from typing import cast

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

sns.set_theme(style="whitegrid")
from pathlib import Path

# 数据文件夹与 notebook 所在文件夹同级。
DATA = Path("../data") if Path("../data").exists() else Path("data")


def check(condition, ok="回答正确！", hint="还不对，请再读一遍题目。"):
    """打印自检单元格的友好结果。"""
    print(("\u2705 " + ok) if condition else ("\u274c " + hint))


print("环境准备完成。数据文件夹：", DATA.resolve())

In [ ]:
dvf_raw = pd.read_csv(DATA / "dvf_gironde_2024.csv.gz", low_memory=False,
                      dtype={"code_commune": str, "id_parcelle": str})
print(dvf_raw.shape, dvf_raw["id_mutation"].nunique(), "transactions")
dvf_raw.head(3)

### 已知的陷阱（还有别的，自己去找）

- 大多数交易有多行：把 `valeur_fonciere` 按行加总会把同一笔价格重复计算好几次。
- 一笔交易可能同时包括一栋住宅**和**一个车库（`Dépendance`），或者好几套公寓：要判断哪些交易是「简单」且可比的。
- 并非每一行都是出售（`Echange` 互换、`Adjudication` 拍卖、`Expropriation` 征收……）。
- 土地和附属建筑缺少面积；有些价格是象征性的（几欧元），有些则高得离谱。
- 每平方米价格分布非常偏斜：用中位数和对数刻度。

### 可能的问题（也可以自己选）

- 波尔多市内和吉伦特省其他地区的独栋住宅与公寓每平方米价格分别是多少（带置信区间）？
- 什么因素解释了价格：面积、房间数、土地、市镇？（对对数价格做 OLS）
- 2024 年的成交数量或价格是否存在季节性规律？
- 能否用 K-Means 按价格水平和住房类型对市镇进行分组？

## 1. 问题

*TODO：写出主问题和两个子问题。它们为什么有用？对谁有用？*

#### Main question
In 2024, in a simple sale of one dwelling in the Gironde — one transaction, one dwelling, between individuals — how much do the surface, the number of rooms, the land sold with it, the type of dwelling (house or flat), and the address contribute to the price? The address is read twice: the distance to the centre of Bordeaux, and the distance to the two holiday destinations of the department, Arcachon and Lacanau-Océan. In other words: what are 10 m², one room, a house and one kilometre worth?
#### Sub question 1
For flats and for houses separately, what is the effect of +10 m² on the median sale price — in euros and in per cent — with a 95 % confidence interval?
#### Sub question 2
Holding the surface, the number of rooms and the type fixed, by what per cent per kilometre does the price per m² fall with the distance to the centre of Bordeaux, and does that gradient weaken beyond 5 km (a quadratic term, and a hinge at exactly 5 km)?

## Why these questions are useful, and for whom
* For a buyer or a seller: the answer prices a dwelling — ten square metres, one room or one kilometre become a number in euros instead of an opinion. DVF is a public register, so every figure can be checked.
* For the public debate: it separates what the market actually pays for (surface, address) from what it does not (the house premium, land measured but not graded), and it measures how fast the address premium fades once a sale is far from the centre.
* For the group itself: sub-question 2 gives the shape of the address effect (linear? convex? a break at 5 km?), which decides how the models of steps 5 and 6 have to be written.
*  For the next team: the same features, cleaned and documented, are the input of the Machine-Learning course of semester 2; a written question tells them what the table is for.


#### 主要研究问题
2024年，在法国吉伦特省（Gironde），对于个人之间一笔仅涉及一套住宅的简单买卖交易，住宅面积、房间数量、随房出售的土地面积、住宅类型（独立屋或公寓）以及地址，各自对成交价格有多大影响？
地址从两个角度衡量：一是到波尔多市中心的距离，二是到省内两个度假目的地——阿卡雄（Arcachon）和拉卡诺海滨（Lacanau-Océan）——的距离。换句话说：10平方米、一个房间、独立屋这一住宅类型，以及一公里的距离，各值多少钱？
#### 子问题1
分别针对公寓和独立屋，住宅面积增加10平方米，对成交价格中位数有何影响？请以欧元金额和百分比表示，并给出95%置信区间。
#### 子问题2
在住宅面积、房间数量和住宅类型保持不变的条件下，到波尔多市中心的距离每增加一公里，每平方米价格下降百分之几？这一价格梯度是否在距离超过5公里后减弱？通过加入距离的二次项，以及在恰好5公里处设置折点的分段线性项来检验。
## 这些问题为何有用，以及对谁有用
* 对买家或卖家而言：研究结果可以为住宅特征定价，让“10平方米”“一个房间”或“一公里”从主观判断变成具体的欧元金额。DVF（法国房地产成交价值数据库）是公开登记数据，因此每个数字都可以核查。
* 对公共讨论而言：研究可以区分市场实际愿意为哪些因素付费（面积、地址），以及哪些因素未体现出相应溢价（独立屋溢价、仅衡量面积而未区分品质的土地），并衡量住宅距离市中心越远，地址溢价消退得有多快。
* 对本小组而言：子问题2揭示地址效应的具体形态：是线性的、凸性的，还是在5公里处出现转折？这将决定第5步和第6步的模型应如何设定。
* 对下一组团队而言：经过清洗并附有说明的同一组特征，将作为第二学期机器学习课程的输入数据；明确写出的研究问题，可以让接手团队理解这张数据表的用途。

## 2. 数据审查

*TODO：一行代表什么？形状、类型、缺失值、重复值、不可能的值、覆盖的时段。复用 TP3 A 部分的 `quality_report`。*

In [ ]:
# 【复用 TP3A】审计第一张表：行数、交易笔数、列数。
n_rows = len(dvf_raw)
n_sales = dvf_raw["id_mutation"].nunique()
per_sale = dvf_raw.groupby("id_mutation").size()

print(f"rows = {n_rows:,}  |  transactions = {n_sales:,}  |  rows per transaction = {n_rows / n_sales:.2f}")
print("\nrows per transaction (number of parts in the same sale):")
print(per_sale.value_counts().sort_index().head(8).to_string())
print("\ndistribution of the number of rows per sale:")
print(per_sale.describe().round(1).to_string())

# 价格在同一笔交易的每一行都重复：直接求和会把价格乘上行数。
assert (dvf_raw.groupby("id_mutation")["valeur_fonciere"].nunique() <= 1).all()
assert per_sale.max() > 10

In [ ]:
def quality_report(df):
    # 【复用 TP3A】审计表：类型、缺失率、唯一值数、一个样例。
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_pct": (100 * df.isna().mean()).round(1),
        "n_unique": df.nunique(),
        "example": [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
    })


report = quality_report(dvf_raw).sort_values("missing_pct", ascending=False)
report

In [ ]:
# 【审计】交易性质的六种取值与行数：只有 Vente 是我们要的市场价。
print("nature of the transaction")
print(dvf_raw["nature_mutation"].value_counts(dropna=False).to_string())

print("\ntype of local (NaN = a plot with no building)")
print(dvf_raw["type_local"].value_counts(dropna=False).to_string())

print("\nfirst and last date registered:", dvf_raw["date_mutation"].min(), "->", dvf_raw["date_mutation"].max())

In [ ]:
exact_dup = int(dvf_raw.duplicated().sum())
same_parcel = int(dvf_raw.duplicated(["id_mutation", "id_parcelle"]).sum())
print(f"|{'exact duplicate rows:':50}| {exact_dup:,}")
print(f"|{'rows sharing (id_mutation, id_parcelle), not dupl.':50}| {same_parcel:,}")

# -- 不可能或可疑的取值 ---------------------------------------------------
print(f"|{'prices <= 10 EUR:':50}| {int(dvf_raw['valeur_fonciere'].le(10).sum()):,}")
print(f"|{'prices > 5 M EUR:':50}| {int(dvf_raw['valeur_fonciere'].gt(5e6).sum()):,}")
print(f"|{'missing price:':50}| {int(dvf_raw['valeur_fonciere'].isna().sum()):,}")
print(f"|{'surface_reelle_bati <= 0 :':50}| {int(dvf_raw['surface_reelle_bati'].le(0).sum()):,}")
print(f"|{'rows without coordinates:':50}| {int(dvf_raw[['longitude', 'latitude']].isna().any(axis=1).sum()):,}")

print("\nprice (EUR):")
print(dvf_raw["valeur_fonciere"].describe(percentiles=[.05, .5, .95]).round(0).to_string())



## 3. 清洗

*TODO：写一个函数 `clean_...(raw)`，把决定写在注释里，之后用 `assert` 检查。*

In [ ]:
# 【常量】Bordeaux Métropole 的 28 个 INSEE 代码（只在稳健性检验里用）。
# Bordeaux Métropole = 28 个市镇（INSEE 代码），来源：https://comersis.fr/epci.php?epci=243300316
METRO_CODES = {
    "33003", "33004", "33013", "33032", "33039", "33056", "33063", "33065", "33075", "33096",
    "33119", "33162", "33167", "33192", "33069", "33200", "33519", "33249", "33273", "33281",
    "33312", "33318", "33376", "33434", "33449", "33487", "33522", "33550",
}
BORDEAUX = "33063"
HABITATION = ["Maison", "Appartement"]
LAT_MIN, LAT_MAX = 44.0, 45.7  # the Gironde in a slightly wider box
LON_MIN, LON_MAX = -1.4, 0.5
PRICE_MIN, PRICE_MAX = 1_000, 5_000_000
PM2_MIN, PM2_MAX = 300, 20_000


def clean_dvf(raw, verbose=True):
    df = raw.copy()
    rows_in = len(df)
    removed = []

    def keep_where(mask, reason):
        nonlocal df
        removed.append((reason, int((~mask).sum())))
        df = df[mask]

    # 1：文件里日期是字符串，转成日期并只保留 2024 年（无缺失日期）。
    df["date_mutation"] = pd.to_datetime(df["date_mutation"], errors="coerce")
    keep_where(df["date_mutation"].notna(), "date is not a date")
    keep_where(df["date_mutation"].dt.year == 2024, "date outside 2024")

    # 2：只有普通买卖才是市场价；期房（尚未建成）、置换、拍卖、征收都剔除。
    keep_where(df["nature_mutation"].eq("Vente"), "not a plain Vente")

    # 3：完全相同的行不含新信息，会让同一观测被算两次。
    keep_where(~df.duplicated(), "exact duplicate row")

    # 4：象征性价格（≤ 10 €）不是市场价；超过 500 万 € 的多是整栋或商业地产，不是一套住宅。
    keep_where(df["valeur_fonciere"].notna(), "price missing")
    keep_where(df["valeur_fonciere"].ge(PRICE_MIN), "symbolic price (< 1,000 EUR)")
    keep_where(df["valeur_fonciere"].le(PRICE_MAX), "estate / commercial price (> 5 M EUR)")

    # 5：位置要用于行政区与距离，所以必须存在且落在吉伦特省框内。
    keep_where(df["latitude"].notna() & df["longitude"].notna(), "D5 coordinates missing")
    keep_where(df["latitude"].between(LAT_MIN, LAT_MAX) & df["longitude"].between(LON_MIN, LON_MAX),
               "coordinates outside the Gironde")

    # 6：研究对象是住宅，因此这笔交易至少要含一套公寓或独栋。
    keep_where(df["id_mutation"].isin(df.loc[df["type_local"].isin(HABITATION), "id_mutation"]),
               "sale without a house or a flat")

    # 7：简单交易 = 恰好一套住宅（房型与面积在住宅行中唯一）
    # （同一套住宅可能因两种土地性质被写两次）；其余行只能是附属物
    # （Dépendance）或纯土地（无房型、无建筑面积）。
    hab = df[df["type_local"].isin(HABITATION)].drop_duplicates(
        ["id_mutation", "type_local", "surface_reelle_bati"])
    signature = hab.groupby("id_mutation").agg(n_type=("type_local", "nunique"),
                                               n_surface=("surface_reelle_bati", "nunique"))
    simple = signature[(signature["n_type"] == 1) & (signature["n_surface"] == 1)].index
    keep_where(df["id_mutation"].isin(simple), "sale with several dwellings")

    other = df[~df["type_local"].isin(HABITATION)]
    tolerated = other["type_local"].eq("Dépendance") | (
            other["type_local"].isna() & other["surface_reelle_bati"].isna())
    keep_where(~df["id_mutation"].isin(other.loc[~tolerated, "id_mutation"]),
               "sale mixed with a commercial or industrial local")

    # 8：住宅；价格是整笔交易的价格（在每一行重复）。
    hab = df[df["type_local"].isin(HABITATION)]
    sales = hab.groupby("id_mutation").agg(
        date_mutation=("date_mutation", "first"),
        valeur_fonciere=("valeur_fonciere", "first"),
        type_local=("type_local", "first"),
        surface_reelle_bati=("surface_reelle_bati", "first"),
        nombre_pieces_principales=("nombre_pieces_principales", "max"),
        nom_commune=("nom_commune", "first"),
        code_commune=("code_commune", "first"),
        code_postal=("code_postal", "first"),
        latitude=("latitude", "first"),
        longitude=("longitude", "first"),
    )
    # 8：土地：先按（地块, 土地性质）取最大值，再在这笔交易内跨地块求和。
    land_rows = df[["id_mutation", "id_parcelle", "nature_culture", "surface_terrain"]].dropna(
        subset=["surface_terrain"]).drop_duplicates()
    land = (land_rows.groupby(["id_mutation", "id_parcelle", "nature_culture"], dropna=False)["surface_terrain"]
            .max().groupby(level="id_mutation").sum())
    dep = df[df["type_local"].eq("Dépendance")].drop_duplicates(
        ["id_mutation", "id_parcelle", "surface_reelle_bati"])
    n_dep = dep.groupby("id_mutation").size()

    sales["code_postal"] = sales["code_postal"].astype("Int64")  # 33290, not 33290.0
    sales["n_dependances"] = n_dep.reindex(sales.index).fillna(0).astype(int)
    sales["surface_terrain_total"] = land.reindex(sales.index)
    sales["nombre_pieces_principales"] = sales["nombre_pieces_principales"].where(
        sales["nombre_pieces_principales"] > 0)  # 0 room with a surface is not a real value

    # 9：算出每 m² 价格，再用一个合理区间筛掉明显错误。
    n_before = len(sales)
    sales = sales[sales["surface_reelle_bati"].gt(0)]  # no division by zero
    removed_sales = [("D9 surface_reelle_bati <= 0", n_before - len(sales))]

    sales["prix_m2"] = sales["valeur_fonciere"] / sales["surface_reelle_bati"]
    sales["log_prix"] = np.log(sales["valeur_fonciere"])
    sales["log_prix_m2"] = np.log(sales["prix_m2"])
    n_before = len(sales)
    sales = sales[sales["prix_m2"].between(PM2_MIN, PM2_MAX)]
    removed_sales.append(("prix_m2 outside [300, 20,000] EUR/m2", n_before - len(sales)))

    # -- 三个行政区（INSEE 代码稳定，市镇名字会变） ---------------------------------
    sales["zone"] = np.where(sales["code_commune"].eq(BORDEAUX), "Bordeaux",
                             np.where(sales["code_commune"].isin(METRO_CODES), "Métropole",
                                      "Reste de la Gironde"))
    sales["month"] = sales["date_mutation"].dt.month

    if verbose:
        print(f"rows kept: {len(df):,} of {rows_in:,}")
        print(pd.DataFrame(removed, columns=["row filter", "rows removed"])
              .set_index("row filter").to_string())
        print("\nsales removed after aggregation:")
        print(pd.DataFrame(removed_sales, columns=["sale filter", "sales removed"])
              .set_index("sale filter").to_string())
        print(f"\nsimple sales kept: {len(sales):,} of {raw['id_mutation'].nunique():,} transactions")

    return sales.reset_index()


############################################################################################################
dvf = clean_dvf(dvf_raw)
dvf.head(3)


#### CHECK THE RESULT 检查结果

In [ ]:
assert dvf["id_mutation"].is_unique  # one row = one sale
assert set(dvf["type_local"]) == set(HABITATION)  # a house or a flat, nothing else
assert dvf["surface_reelle_bati"].gt(0).all()  # no zero surface before dividing
assert dvf["valeur_fonciere"].ge(PRICE_MIN).all()
assert dvf["valeur_fonciere"].le(PRICE_MAX).all()
assert dvf["date_mutation"].dt.year.eq(2024).all()
assert dvf["latitude"].between(LAT_MIN, LAT_MAX).all()
assert dvf["longitude"].between(LON_MIN, LON_MAX).all()
assert dvf["prix_m2"].between(PM2_MIN, PM2_MAX).all()
assert set(dvf["zone"]) == {"Bordeaux", "Métropole", "Reste de la Gironde"}
assert dvf[["valeur_fonciere", "surface_reelle_bati", "type_local", "zone", "date_mutation"]].notna().all().all()
assert dvf["n_dependances"].ge(0).all()

print(f"{len(dvf):,} simple sales | {len(dvf) / dvf_raw['id_mutation'].nunique():.0%} of the transactions of the file")
print(
    f"median price {dvf['valeur_fonciere'].median():,.0f} EUR | median surface {dvf['surface_reelle_bati'].median():.0f} m2")
dvf[["valeur_fonciere", "surface_reelle_bati", "nombre_pieces_principales", "surface_terrain_total",
     "n_dependances", "prix_m2"]].describe(percentiles=[.5, .95]).round(1)

## 4. 探索

*TODO：至少五张图，每张图后面跟一句话。*

In [ ]:
# 【复用 TP2A】特征工程：对数面积、土地，以及到佩贝朗广场的向量化 haversine 距离；N_BOOT 也在这里。
FIG = Path("figures")
FIG.mkdir(exist_ok=True)
N_BOOT = 1000

# -- 住宅 ---------------------------------------------------------------------
dvf["type_local"] = dvf["type_local"].astype(str)
dvf["is_maison"] = dvf["type_local"].eq("Maison").astype(int)
dvf["log_surface"] = np.log(dvf["surface_reelle_bati"])
dvf["land_m2"] = dvf["surface_terrain_total"].fillna(0)
dvf["has_land"] = dvf["surface_terrain_total"].notna().astype(int)
dvf["log_land"] = np.log1p(dvf["land_m2"])

# -- 把地址压缩成一个数：到波尔多市中心的距离 -----------------------------------
# 中心点 = 佩贝朗广场（市政厅），北纬 44.8378 / 东经 −0.5792。
# 距离 = 大圆距离（haversine，单位 km）：忽略道路、桥梁与交通，这是局限
# （第 8 步写明）；但这是文件唯一能支撑的距离，而且每一笔都能算。
CENTRE_LAT, CENTRE_LON, EARTH_KM = 44.8378, -0.5792, 6371.0


def haversine_km(lat, lon, lat0=CENTRE_LAT, lon0=CENTRE_LON):
    # 球面两点的大圆距离：2R·arcsin(√a)。
    lat, lon = np.radians(lat), np.radians(lon)
    lat0, lon0 = np.radians(lat0), np.radians(lon0)
    a = np.sin((lat - lat0) / 2) ** 2 + np.cos(lat) * np.cos(lat0) * np.sin((lon - lon0) / 2) ** 2
    return 2 * EARTH_KM * np.atan(np.sqrt(a), np.sqrt(1 - a))


dvf["distance_km"] = haversine_km(dvf["latitude"].to_numpy(), dvf["longitude"].to_numpy())
dvf["distance_km_sq"] = dvf["distance_km"] ** 2
dvf["km_beyond_5"] = (dvf["distance_km"] - 5).clip(lower=0)  # hinge: 0 inside 5 km, km past it after

assert dvf["distance_km"].between(0, 120).all()  # a sale in the Gironde is within 120 km of Pey Berland
assert dvf[["log_surface", "distance_km", "log_prix"]].notna().all().all()
print(f"{len(dvf):,} simple sales of 2024")
print(f"distance to the centre: median {dvf['distance_km'].median():.1f} km"
      f" | 50 % of the sales between {dvf['distance_km'].quantile(0.25):.1f} and {dvf['distance_km'].quantile(0.75):.1f} km"
      f" | 95 % under {dvf['distance_km'].quantile(0.95):.0f} km | max {dvf['distance_km'].max():.0f} km")
print(f"houses {dvf['is_maison'].mean():.0%} | flats {1 - dvf['is_maison'].mean():.0%}"
      f" | with land recorded {dvf['has_land'].mean():.0%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for ax, kind in zip(axes, ["Appartement", "Maison"]):
    x = dvf.loc[dvf["type_local"].eq(kind), "prix_m2"]
    bins = np.geomspace(300, x.quantile(0.999), 45)
    ax.hist(x, bins=bins, color="#4C72B0" if kind == "Appartement" else "#DD8452", alpha=0.85)
    ax.set_xscale("log")
    ax.axvline(x.median(), color="black", lw=1.6, label=f"median {x.median():,.0f} EUR/m2")
    ax.axvline(x.mean(), color="black", lw=1.2, ls="--", label=f"mean {x.mean():,.0f} EUR/m2")
    ax.set_title(f"{kind} ({len(x):,} sales)")
    ax.set_xlabel("price per m2 (EUR, log scale)")
    ax.legend(fontsize=8)
axes[0].set_ylabel("number of sales")
fig.suptitle("A long right tail: the mean is above the median in both markets", y=1.02)
fig.tight_layout()
fig.savefig(FIG / "fig1_pm2_distribution.png", dpi=150, bbox_inches="tight")

In [ ]:
# 【复用 TP3B】图 2：每 m² 中位价随距离的变化（按房型分线）+ 每个距离带的成交量。
edges = [0, 2, 4, 6, 8, 12, 16, 20, 25, 30, 40, 200]
d = dvf.copy()
d["dist_bin"] = pd.cut(d["distance_km"], edges)
tab = d.pivot_table(index="dist_bin", columns="type_local", values="prix_m2", aggfunc="median", observed=True)
cnt = d.pivot_table(index="dist_bin", columns="type_local", values="prix_m2", aggfunc="size", observed=True)
shown = tab.where(cnt >= 20)  # a bin with fewer than 20 sales says nothing
x = [(iv.left + iv.right) / 2 for iv in tab.index]
x[-1] = 55.0  # the open top bin
labels = [f"{int(iv.left)}-{int(iv.right)}" if iv.right < 200 else f"{int(iv.left)}+" for iv in tab.index]

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.4), gridspec_kw={"width_ratios": [1.55, 1]})
ax = axes[0]
for kind, color in [("Appartement", "#4C72B0"), ("Maison", "#DD8452")]:
    ax.plot(x, shown[kind].values, marker="o", color=color, label=kind)
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=45, fontsize=7.5)
ax.set_yscale("log")
ax.set_xlabel("distance to the centre of Bordeaux (km)")
ax.set_ylabel("median price per m2 (EUR, log scale)")
ax.set_title("Price per m2 against the distance to the centre")
ax.legend(fontsize=8)

bx = np.arange(len(labels))
axes[1].bar(bx - 0.2, cnt["Appartement"].fillna(0).values, width=0.4, color="#4C72B0", label="Appartement")
axes[1].bar(bx + 0.2, cnt["Maison"].fillna(0).values, width=0.4, color="#DD8452", label="Maison")
axes[1].set_xticks(bx)
axes[1].set_xticklabels(labels, rotation=45, fontsize=7.5)
axes[1].set_yscale("log")
axes[1].set_xlabel("distance to the centre (km)")
axes[1].set_ylabel("number of sales (log scale)")
axes[1].set_title("Where the sales are")
axes[1].legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG / "fig2_distance_profile.png", dpi=150, bbox_inches="tight")
print((shown.round(0).to_string()))

In [ ]:
# 【复用 TP3B】图 3：双对数坐标下的价格–面积关系（斜率 ≈ 0.7–0.8）。
fig, ax = plt.subplots(figsize=(7.2, 5.2))
for kind, color in [("Appartement", "#4C72B0"), ("Maison", "#DD8452")]:
    s = dvf[dvf["type_local"].eq(kind)]
    ax.scatter(s["surface_reelle_bati"], s["valeur_fonciere"], s=6, alpha=0.25, color=color, label=kind)
    grid = np.quantile(s["surface_reelle_bati"], np.linspace(0.05, 0.95, 12))
    med = s.groupby(pd.cut(s["surface_reelle_bati"], np.r_[0, grid, np.inf]), observed=True)["valeur_fonciere"].median()
    ax.plot([iv.mid for iv in med.index], med.values, color=color, lw=2.4, marker="o", ms=3)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("surface (m2, log scale)")
ax.set_ylabel("sale price (EUR, log scale)")
ax.set_title("Price against surface: a straight line on a log-log plane")
ax.legend()
fig.tight_layout()
fig.savefig(FIG / "fig3_surface_price.png", dpi=150, bbox_inches="tight")

In [ ]:
# 【复用 TP3B】图 5：以佩贝朗广场为中心的地图，颜色 = 每 m² 价，环 = 5 / 15 km。
# 以南佩贝朗广场为原点（东西 / 南北，单位 km）重新画的地图：环就是模型用的距离。
lat0, lon0 = CENTRE_LAT, CENTRE_LON
kx = (dvf["longitude"] - lon0) * 111.32 * np.cos(np.radians(lat0))
ky = (dvf["latitude"] - lat0) * 110.57
fig, ax = plt.subplots(figsize=(7.8, 6.4))
sc = ax.scatter(kx, ky, c=dvf["prix_m2"], s=5, alpha=0.35, cmap="magma_r", vmin=1000, vmax=7000, label="sale")
pl = plt.colorbar(sc, ax=ax, shrink=0.9)
pl.set_label("price per m2 (EUR)")
for r in [5, 10, 20, 40]:
    ax.add_patch(plt.Circle((0, 0), r, fill=False, color="#333333", lw=0.9, ls="--"))
    ax.text(r * 0.71, r * 0.71, f"{r} km", fontsize=8, color="#333333", ha="center", va="center")
ax.plot(0, 0, marker="*", color="black", ms=14)
ax.text(0.6, -2.2, "Place Pey Berland", fontsize=8)
ax.set_aspect("equal")
ax.set_xlabel("km east of the centre")
ax.set_ylabel("km north of the centre")
ax.set_title("The Gironde around Bordeaux: the dashed rings are the distance used in the model")
fig.tight_layout()
fig.savefig(FIG / "fig5_map_distance.png", dpi=150, bbox_inches="tight")

## 5. 统计

*TODO：一个置信区间和一个检验或 OLS 回归，效应量以真实单位表示。*

In [ ]:
# TODO（在此处完成）

## 6. 可直接用于机器学习的数据集

*TODO：观测单位、目标、特征、训练/测试划分、只在训练集上做标准化、导出 CSV + 数据字典。建议的观测单位：一笔简单交易（一栋住宅或一套公寓）；目标：每平方米价格（或对数价格）；特征：面积、房间数、土地、类型、位置；随机划分可以接受（只有一年的数据）。*

In [ ]:
# TODO（在此处完成）

## 7. 基线

*TODO：一个朴素规则、感知机或 K-Means（TP4）、一个指标，以及一段解读。*

In [ ]:
# TODO（在此处完成）

## 8. 结论与局限

*TODO：回答问题、局限、后续步骤。*

In [ ]:
# TODO（在此处完成）